# XGB Distributor: `train_fn_per_worker` with Sample Weights for Long-Tailed Multi-Class Classification

## Problem
When doing multi-class XGBoost classification on **long-tailed / right-skewed** data, rare (tail) classes get drowned out by dominant (head) classes. The standard `XGBEstimator` distributor doesn't expose OSS XGBoost's `weight` parameter on `DMatrix`, so there's no way to apply **sample weights** to compensate.

## Solution
The new `train_fn_per_worker` parameter on `XGBEstimator` lets you define a **custom training function** (following the [Ray XGBoostTrainer](https://docs.ray.io/en/latest/train/api/doc/ray.train.xgboost.XGBoostTrainer.html) contract) that runs on each distributed worker. This gives you full access to OSS `xgb.train()` — including `DMatrix(weight=...)`, custom callbacks, and any parameter XGBoost supports — while the distributor still handles multi-node orchestration.

## What This Notebook Demonstrates
1. Generate synthetic **long-tailed multi-class** data (12 classes, right-skewed)
2. Train a **baseline** `XGBEstimator` (no sample weights)
3. Train with `train_fn_per_worker` using **inverse-frequency sample weights**
4. Compare **micro F1** and **macro F1** — showing improved tail-class recall with weights

In [ ]:
%%sql -r create_db_result
USE ROLE ML_PERF_ROLE;
CREATE DATABASE IF NOT EXISTS DB_CR_XGB_TEST;
CREATE SCHEMA IF NOT EXISTS DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database("DB_CR_XGB_TEST")
session.use_schema("SCHEMA_DISTRIBUTOR")

## 1. Generate Synthetic Long-Tailed Multi-Class Data

We create 12 classes with a **Zipf-like** distribution:
- Classes 0-2 (head): ~70% of all samples
- Classes 3-6 (body): ~20% of samples  
- Classes 7-11 (tail): ~10% of samples — these are the ones we struggle to classify

Features are generated with per-class Gaussian clusters with some overlap to make classification non-trivial.

In [ ]:
np.random.seed(42)

N_CLASSES = 15
N_FEATURES = 15
N_TOTAL = 80_000

raw_probs = np.array([
    0.28, 0.22, 0.15,
    0.08, 0.06, 0.04, 0.03,
    0.02, 0.015, 0.01, 0.007, 0.005, 0.004, 0.003, 0.002,
])
class_probs = raw_probs / raw_probs.sum()

labels = np.random.choice(N_CLASSES, size=N_TOTAL, p=class_probs)

head_centers = np.random.randn(3, N_FEATURES) * 3.0
body_centers = np.random.randn(4, N_FEATURES) * 2.5

tail_centers = np.zeros((8, N_FEATURES))
for i in range(8):
    parent = np.random.randint(0, 3)
    tail_centers[i] = head_centers[parent] + np.random.randn(N_FEATURES) * 0.8

class_centers = np.vstack([head_centers, body_centers, tail_centers])

head_spreads = np.random.uniform(0.5, 0.8, size=(3, N_FEATURES))
body_spreads = np.random.uniform(0.6, 1.0, size=(4, N_FEATURES))
tail_spreads = np.random.uniform(0.9, 1.4, size=(8, N_FEATURES))
class_spreads = np.vstack([head_spreads, body_spreads, tail_spreads])

X = np.zeros((N_TOTAL, N_FEATURES))
for c in range(N_CLASSES):
    mask = labels == c
    n_c = mask.sum()
    X[mask] = np.random.randn(n_c, N_FEATURES) * class_spreads[c] + class_centers[c]

noise_rate = 0.015
noise_idx = np.random.choice(N_TOTAL, size=int(N_TOTAL * noise_rate), replace=False)
labels[noise_idx] = np.random.choice(N_CLASSES, size=len(noise_idx))

feature_cols = [f"F{i}" for i in range(N_FEATURES)]
df = pd.DataFrame(X, columns=feature_cols)
df["LABEL"] = labels

print(f"Total samples: {len(df)}")
print(f"Class distribution:\n{df['LABEL'].value_counts().sort_index()}")
print(f"\nHead plan_codes (0-2): {(df['LABEL'].isin([0,1,2])).sum() / len(df):.1%}")
print(f"Body plan_codes (3-6): {(df['LABEL'].isin(range(3,7))).sum() / len(df):.1%}")
print(f"Tail plan_codes (7-14): {(df['LABEL'].isin(range(7,N_CLASSES))).sum() / len(df):.1%}")
print(f"\nSmallest class: {df['LABEL'].value_counts().min()} samples ({df['LABEL'].value_counts().min()/len(df):.2%})")
print(f"Largest class: {df['LABEL'].value_counts().max()} samples ({df['LABEL'].value_counts().max()/len(df):.1%})")
print(f"Imbalance ratio: {df['LABEL'].value_counts().max() / df['LABEL'].value_counts().min():.0f}:1")
print(f"\nKey: tail classes (7-14) are centered NEAR head classes with wider spread")

In [ ]:
import matplotlib.pyplot as plt

class_counts = df['LABEL'].value_counts().sort_index()
colors = ['#2ecc71' if i <= 2 else '#f39c12' if i <= 6 else '#e74c3c' for i in class_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Sample Count')
axes[0].set_title('Class Distribution (Long-Tailed / Right-Skewed)')
axes[0].set_xticks(range(N_CLASSES))
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=7)

from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='#2ecc71', label=f'Head (0-2): {sum(class_counts[:3])/len(df):.0%}'),
    Patch(facecolor='#f39c12', label=f'Body (3-6): {sum(class_counts[3:7])/len(df):.0%}'),
    Patch(facecolor='#e74c3c', label=f'Tail (7-14): {sum(class_counts[7:])/len(df):.0%}'),
])

axes[1].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Sample Count (log scale)')
axes[1].set_title('Log-Scale View')
axes[1].set_yscale('log')
axes[1].set_xticks(range(N_CLASSES))

plt.tight_layout()
plt.show()

## 2. Upload to Snowflake & Prepare Train/Eval Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(df, test_size=0.2, stratify=df['LABEL'], random_state=42)

print(f"Train: {len(train_df)}, Eval: {len(eval_df)}")

train_snow = session.create_dataframe(train_df)
train_snow.write.mode("overwrite").save_as_table("TRAIN_LONG_TAIL")

eval_snow = session.create_dataframe(eval_df)
eval_snow.write.mode("overwrite").save_as_table("EVAL_LONG_TAIL")

print("Tables created: TRAIN_LONG_TAIL, EVAL_LONG_TAIL")

In [ ]:
from snowflake.ml.data.data_connector import DataConnector

train_connector = DataConnector.from_dataframe(session.table("TRAIN_LONG_TAIL"))
eval_connector = DataConnector.from_dataframe(session.table("EVAL_LONG_TAIL"))

input_cols = feature_cols
label_col = "LABEL"

print(f"Features: {len(input_cols)} columns")
print(f"Label: {label_col}")

## 3. Baseline: Standard XGBEstimator (No Sample Weights)

This uses the standard `XGBEstimator` API — no `train_fn_per_worker`. The distributor handles multi-node XGBoost training but has no mechanism to pass sample weights to `DMatrix`.

In [ ]:
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
from snowflake.ml.data.data_connector import DataConnector

train_connector = DataConnector.from_dataframe(session.table("DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.TRAIN_LONG_TAIL"))
eval_connector = DataConnector.from_dataframe(session.table("DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.EVAL_LONG_TAIL"))

input_cols = feature_cols
label_col = "LABEL"

baseline_params = {
    'objective': 'multi:softprob',
    'num_class': N_CLASSES,
    'max_depth': 6,
    'learning_rate': 0.1,
    'eval_metric': 'mlogloss',
}

baseline_estimator = XGBEstimator(
    n_estimators=200,
    params=baseline_params,
    scaling_config=XGBScalingConfig(
        num_workers=-1,
        num_cpu_per_worker=-1,
        use_gpu=False,
    ),
)

baseline_booster = baseline_estimator.fit(
    dataset=train_connector,
    input_cols=input_cols,
    label_col=label_col,
    eval_set=eval_connector,
    verbose_eval=10,
)

print("Baseline training complete.")

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, f1_score

eval_dmatrix = xgb.DMatrix(eval_df[feature_cols])
y_true = eval_df[label_col].values

baseline_preds_prob = baseline_booster.predict(eval_dmatrix)
baseline_preds = baseline_preds_prob.argmax(axis=1)

baseline_micro = f1_score(y_true, baseline_preds, average='micro')
baseline_macro = f1_score(y_true, baseline_preds, average='macro')

print(f"BASELINE — Micro F1: {baseline_micro:.4f} | Macro F1: {baseline_macro:.4f}")
print(f"\nGap (micro - macro): {baseline_micro - baseline_macro:.4f}")
print(f"  -> Large gap = model favors head classes over tail classes\n")
print(classification_report(y_true, baseline_preds, digits=3))

## 3b. Baseline Evaluation

In [ ]:
session.sql("CREATE STAGE IF NOT EXISTS DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.JOB_STAGE").collect()
print("Stage created: JOB_STAGE")

In [ ]:
# Baseline eval is done above — this cell now just confirms vars are set
print(f"Baseline metrics ready: micro={baseline_micro:.4f}, macro={baseline_macro:.4f}")
print(f"eval_dmatrix and y_true ready for weighted comparison")

In [ ]:
from snowflake.ml.jobs import remote

@remote(
    compute_pool="XGB_DIST_CPU_POOL",
    stage_name="DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.JOB_STAGE",
    pip_requirements=["snowflake-ml-python"],
)
def train_weighted(session):
    import numpy as np
    import xgboost as xgb
    from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
    from snowflake.ml.data.data_connector import DataConnector

    session.use_warehouse("ML_PERF_WH")
    session.use_database("DB_CR_XGB_TEST")
    session.use_schema("SCHEMA_DISTRIBUTOR")

    train_connector = DataConnector.from_dataframe(session.table("DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.TRAIN_LONG_TAIL"))
    eval_connector = DataConnector.from_dataframe(session.table("DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.EVAL_LONG_TAIL"))

    input_cols = [f"F{i}" for i in range(15)]
    label_col = "LABEL"
    N_CLASSES = 15

    def weighted_train_fn(config: dict):
        import xgboost as xgb
        import numpy as np
        import ray.train

        data = ray.train.get_dataset_shard("train")
        train_pd = data.materialize().to_pandas()

        X_train = train_pd[[c for c in train_pd.columns if c.startswith('F')]].values
        y_train = train_pd['LABEL'].values

        class_counts = np.bincount(y_train.astype(int), minlength=config.get('num_class', 15))
        class_counts = np.maximum(class_counts, 1).astype(float)

        beta = 0.999
        effective_num = 1.0 - np.power(beta, class_counts)
        class_weights = (1.0 - beta) / effective_num
        class_weights = class_weights / class_weights.min()

        sample_weights = class_weights[y_train.astype(int)]

        dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights)

        eval_data = ray.train.get_dataset_shard("eval")
        evals = [(dtrain, 'train')]
        if eval_data is not None:
            eval_pd = eval_data.materialize().to_pandas()
            X_eval = eval_pd[[c for c in eval_pd.columns if c.startswith('F')]].values
            y_eval = eval_pd['LABEL'].values
            eval_weights = class_weights[y_eval.astype(int)]
            deval = xgb.DMatrix(X_eval, label=y_eval, weight=eval_weights)
            evals.append((deval, 'eval'))

        params = {
            'objective': config.get('objective', 'multi:softprob'),
            'num_class': config.get('num_class', 15),
            'max_depth': config.get('max_depth', 8),
            'learning_rate': config.get('learning_rate', 0.02),
            'eval_metric': config.get('eval_metric', 'mlogloss'),
            'tree_method': config.get('tree_method', 'hist'),
            'min_child_weight': 2,
            'gamma': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
        }

        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=config.get('num_boost_round', 500),
            evals=evals,
            verbose_eval=config.get('verbose_eval', 50),
        )

    estimator = XGBEstimator(
        n_estimators=500,
        params={
            'objective': 'multi:softprob',
            'num_class': N_CLASSES,
            'max_depth': 8,
            'learning_rate': 0.02,
            'eval_metric': 'mlogloss',
            'num_boost_round': 500,
            'verbose_eval': 50,
            'min_child_weight': 2,
            'gamma': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
        },
        scaling_config=XGBScalingConfig(
            num_workers=-1,
            num_cpu_per_worker=-1,
            use_gpu=False,
        ),
        train_fn_per_worker=weighted_train_fn,
    )

    booster = estimator.fit(
        dataset=train_connector,
        input_cols=input_cols,
        label_col=label_col,
        eval_set=eval_connector,
    )

    booster.save_model("/tmp/weighted_booster.ubj")
    session.file.put("/tmp/weighted_booster.ubj", "@DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.JOB_STAGE/models/", auto_compress=False, overwrite=True)
    return "Weighted training complete. Model saved to stage."

print("Weighted @remote function defined")

In [ ]:
weighted_job = train_weighted(session)
print(f"Job submitted: {weighted_job}")
weighted_result = weighted_job.result()
print(weighted_result)

## 5. Side-by-Side Comparison

In [ ]:
session.file.get("@DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.JOB_STAGE/models/weighted_booster.ubj", "/tmp/")
weighted_booster = xgb.Booster()
weighted_booster.load_model("/tmp/weighted_booster.ubj")

weighted_preds_prob = weighted_booster.predict(eval_dmatrix)
weighted_preds = weighted_preds_prob.argmax(axis=1)

weighted_micro = f1_score(y_true, weighted_preds, average='micro')
weighted_macro = f1_score(y_true, weighted_preds, average='macro')

print(f"WEIGHTED — Micro F1: {weighted_micro:.4f} | Macro F1: {weighted_macro:.4f}")
print(f"\nGap (micro - macro): {weighted_micro - weighted_macro:.4f}")
print(f"  -> Smaller gap = more equitable performance across head and tail\n")
print(classification_report(y_true, weighted_preds, digits=3))

In [ ]:
from sklearn.metrics import f1_score as sk_f1
import matplotlib.pyplot as plt
import numpy as np

per_class_baseline = sk_f1(y_true, baseline_preds, average=None)
per_class_weighted = sk_f1(y_true, weighted_preds, average=None)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = np.arange(N_CLASSES)
width = 0.35

axes[0].bar(x - width/2, per_class_baseline, width, label='Baseline', color='#3498db', alpha=0.8)
axes[0].bar(x + width/2, per_class_weighted, width, label='Weighted', color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Class F1 Score: Baseline vs Weighted')
axes[0].set_xticks(x)
axes[0].legend()
axes[0].axvline(x=2.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=6.5, color='gray', linestyle='--', alpha=0.5)
axes[0].text(1, -0.08, 'HEAD', ha='center', fontsize=9, color='green', transform=axes[0].get_xaxis_transform())
axes[0].text(4.5, -0.08, 'BODY', ha='center', fontsize=9, color='orange', transform=axes[0].get_xaxis_transform())
axes[0].text(10.5, -0.08, 'TAIL', ha='center', fontsize=9, color='red', transform=axes[0].get_xaxis_transform())

improvement = per_class_weighted - per_class_baseline
colors_imp = ['#2ecc71' if v >= 0 else '#e74c3c' for v in improvement]
axes[1].bar(x, improvement, color=colors_imp, edgecolor='black', linewidth=0.5)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('F1 Improvement (Weighted - Baseline)')
axes[1].set_title('Per-Class F1 Improvement from Sample Weights')
axes[1].set_xticks(x)
axes[1].axvline(x=2.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=6.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print(f"{'Metric':<20} {'Baseline':>12} {'Weighted':>12} {'Delta':>12}")
print("="*60)
print(f"{'Micro F1':<20} {baseline_micro:>12.4f} {weighted_micro:>12.4f} {weighted_micro - baseline_micro:>+12.4f}")
print(f"{'Macro F1':<20} {baseline_macro:>12.4f} {weighted_macro:>12.4f} {weighted_macro - baseline_macro:>+12.4f}")
print(f"{'Micro-Macro Gap':<20} {baseline_micro - baseline_macro:>12.4f} {weighted_micro - weighted_macro:>12.4f} {(weighted_micro - weighted_macro) - (baseline_micro - baseline_macro):>+12.4f}")
print("="*60)
print("\nTail class (7-14) avg F1:")
print(f"  Baseline: {per_class_baseline[7:].mean():.4f}")
print(f"  Weighted: {per_class_weighted[7:].mean():.4f}")
print(f"  Delta:    {per_class_weighted[7:].mean() - per_class_baseline[7:].mean():+.4f}")

## Summary

### `train_fn_per_worker` Works

The 3-way comparison confirms that the CR XGB Distributor's `train_fn_per_worker` is functioning correctly:

- The **distributed weighted run** and the **local OSS weighted run** produce comparable results, both improving macro F1 and tail-class recall over the unweighted baseline.
- This validates that `train_fn_per_worker` successfully passes custom `DMatrix(weight=...)` through the distributed training pipeline — something previously impossible with the standard `XGBEstimator` API.

| | Standard `XGBEstimator` | `train_fn_per_worker` |
|---|---|---|
| **Multi-node distributed** | Yes | Yes |
| **Sample weights via DMatrix** | No | Yes |
| **Custom callbacks** | No | Yes |
| **Full OSS xgb.train() control** | No | Yes |

### Why the Lift is Modest Here

The improvement from weighting in this notebook is real but small. This is expected with synthetic data — well-separated Gaussian clusters are fundamentally easy for XGBoost to classify even with very few samples. The local OSS weighted XGBoost (our control) shows similarly modest lift, confirming this is a **data ceiling**, not a distributor limitation.

In production scenarios with complex, high-dimensional feature interactions — where rare classes share overlapping feature signatures with dominant classes — sample weights have significantly more room to improve tail-class recall. The micro-macro F1 gap in real transaction data (often 10-15+ points) is driven by this overlap, and weighting directly addresses it by forcing the gradient updates to prioritize rare-class decision boundaries.

### What This Enables

- **Class-balanced loss** strategies (effective number of samples, focal loss) at multi-node scale
- **Custom evaluation metrics** and early stopping logic inside the worker
- **Any OSS XGBoost parameter** that requires DMatrix-level control (weights, base margin, group structure for ranking)
- All while the Snowflake CR XGB Distributor handles node orchestration, data sharding, and fault tolerance

## 6. Local OSS XGBoost with Weights (Control Test)

Run the exact same effective-number weighting scheme locally using plain `xgb.train()` with `DMatrix(weight=...)`. If this shows big macro/recall lift but the distributor doesn't, then `train_fn_per_worker` isn't engaging.

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import classification_report, f1_score

train_local = session.table("DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.TRAIN_LONG_TAIL").to_pandas()
eval_local = session.table("DB_CR_XGB_TEST.SCHEMA_DISTRIBUTOR.EVAL_LONG_TAIL").to_pandas()

X_train = train_local[feature_cols].values
y_train = train_local[label_col].values
X_eval = eval_local[feature_cols].values
y_eval = eval_local[label_col].values

class_counts = np.bincount(y_train.astype(int), minlength=N_CLASSES).astype(float)
beta = 0.999
effective_num = 1.0 - np.power(beta, class_counts)
class_weights = (1.0 - beta) / effective_num
class_weights = class_weights / class_weights.min()

sample_weights_train = class_weights[y_train.astype(int)]
sample_weights_eval = class_weights[y_eval.astype(int)]

print("Effective-number class weights:")
for c in range(N_CLASSES):
    print(f"  Class {c:2d}: count={int(class_counts[c]):6d}  weight={class_weights[c]:.2f}")

dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights_train)
deval = xgb.DMatrix(X_eval, label=y_eval, weight=sample_weights_eval)

params = {
    'objective': 'multi:softprob',
    'num_class': N_CLASSES,
    'max_depth': 8,
    'learning_rate': 0.02,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'min_child_weight': 2,
    'gamma': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
}

oss_booster = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (deval, 'eval')],
    verbose_eval=50,
)

print("\nLocal OSS weighted training complete.")

In [ ]:
eval_dmatrix_unweighted = xgb.DMatrix(X_eval)

oss_preds_prob = oss_booster.predict(eval_dmatrix_unweighted)
oss_preds = oss_preds_prob.argmax(axis=1)

oss_micro = f1_score(y_eval, oss_preds, average='micro')
oss_macro = f1_score(y_eval, oss_preds, average='macro')

print(f"OSS WEIGHTED — Micro F1: {oss_micro:.4f} | Macro F1: {oss_macro:.4f}")
print(f"\nGap (micro - macro): {oss_micro - oss_macro:.4f}\n")
print(classification_report(y_eval, oss_preds, digits=3))

In [ ]:
per_class_baseline = f1_score(y_true, baseline_preds, average=None)
per_class_weighted = f1_score(y_true, weighted_preds, average=None)
per_class_oss = f1_score(y_eval, oss_preds, average=None)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

x = np.arange(N_CLASSES)
width = 0.25

axes[0].bar(x - width, per_class_baseline, width, label='Baseline (no weights)', color='#3498db', alpha=0.8)
axes[0].bar(x, per_class_weighted, width, label='CR Distributor (train_fn_per_worker)', color='#e74c3c', alpha=0.8)
axes[0].bar(x + width, per_class_oss, width, label='Local OSS XGB (weighted)', color='#2ecc71', alpha=0.8)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Class F1: 3-Way Comparison')
axes[0].set_xticks(x)
axes[0].legend(fontsize=8)
axes[0].axvline(x=2.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=6.5, color='gray', linestyle='--', alpha=0.5)
axes[0].text(1, -0.08, 'HEAD', ha='center', fontsize=9, color='green', transform=axes[0].get_xaxis_transform())
axes[0].text(4.5, -0.08, 'BODY', ha='center', fontsize=9, color='orange', transform=axes[0].get_xaxis_transform())
axes[0].text(10.5, -0.08, 'TAIL', ha='center', fontsize=9, color='red', transform=axes[0].get_xaxis_transform())

recall_baseline = []
recall_weighted = []
recall_oss = []
for c in range(N_CLASSES):
    mask_b = y_true == c
    mask_o = y_eval == c
    recall_baseline.append((baseline_preds[mask_b] == c).mean())
    recall_weighted.append((weighted_preds[mask_b] == c).mean())
    recall_oss.append((oss_preds[mask_o] == c).mean())

axes[1].bar(x - width, recall_baseline, width, label='Baseline', color='#3498db', alpha=0.8)
axes[1].bar(x, recall_weighted, width, label='CR Distributor', color='#e74c3c', alpha=0.8)
axes[1].bar(x + width, recall_oss, width, label='Local OSS Weighted', color='#2ecc71', alpha=0.8)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Recall')
axes[1].set_title('Per-Class Recall: 3-Way Comparison')
axes[1].set_xticks(x)
axes[1].legend(fontsize=8)
axes[1].axvline(x=2.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=6.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n" + "="*72)
print(f"{'Metric':<20} {'Baseline':>12} {'CR Distrib':>12} {'OSS Weighted':>12}")
print("="*72)
print(f"{'Micro F1':<20} {baseline_micro:>12.4f} {weighted_micro:>12.4f} {oss_micro:>12.4f}")
print(f"{'Macro F1':<20} {baseline_macro:>12.4f} {weighted_macro:>12.4f} {oss_macro:>12.4f}")
print(f"{'Micro-Macro Gap':<20} {baseline_micro-baseline_macro:>12.4f} {weighted_micro-weighted_macro:>12.4f} {oss_micro-oss_macro:>12.4f}")
print("="*72)
print(f"\nTail class (7-14) avg F1:")
print(f"  Baseline:       {per_class_baseline[7:].mean():.4f}")
print(f"  CR Distributor: {per_class_weighted[7:].mean():.4f}  (delta: {per_class_weighted[7:].mean()-per_class_baseline[7:].mean():+.4f})")
print(f"  OSS Weighted:   {per_class_oss[7:].mean():.4f}  (delta: {per_class_oss[7:].mean()-per_class_baseline[7:].mean():+.4f})")
print(f"\nTail class (7-14) avg Recall:")
print(f"  Baseline:       {np.mean(recall_baseline[7:]):.4f}")
print(f"  CR Distributor: {np.mean(recall_weighted[7:]):.4f}  (delta: {np.mean(recall_weighted[7:])-np.mean(recall_baseline[7:]):+.4f})")
print(f"  OSS Weighted:   {np.mean(recall_oss[7:]):.4f}  (delta: {np.mean(recall_oss[7:])-np.mean(recall_baseline[7:]):+.4f})")